In [1]:
import torch

import copy
from collections import OrderedDict

In [2]:
inputs=torch.tensor( # 6 tokens long sentence, each token is 3D embedding
    [[0.43,0.15,0.89],
     [0.55,0.87,0.66],
     [0.57,0.85,0.64],
     [0.22,0.58,0.33],
     [0.77,0.25,0.10],
     [0.05,0.80,0.55]]
)
print(f"{inputs.shape}")

torch.Size([6, 3])


In [3]:
x_2=inputs[1]
d_in=inputs.shape[1]
d_out=2

print(f"{x_2.shape=}")

# initialize 3 weight matrices
torch.manual_seed(123)
W_query=torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key=torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value=torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

query_2=x_2@W_query # where x_2 acts as if it is of size [1,3]
key_2=x_2@W_key
value_2=x_2@W_value
print(f"{query_2.shape=}, {key_2.shape=}, {value_2.shape=}")
print(f"{query_2=}")

x_2.shape=torch.Size([3])
query_2.shape=torch.Size([2]), key_2.shape=torch.Size([2]), value_2.shape=torch.Size([2])
query_2=tensor([0.4306, 1.4551])


In [4]:
keys=inputs@W_key
values=inputs@W_value
queries=inputs@W_query
print(f"{keys.shape=}, {values.shape=}, {queries.shape=}")

keys.shape=torch.Size([6, 2]), values.shape=torch.Size([6, 2]), queries.shape=torch.Size([6, 2])


In [5]:
# attention scores are the multiplication of query and keys
attention_scores=queries @ keys.T
print(f"{attention_scores.shape=}\n{attention_scores}")

d_k=keys.shape[-1]
print(f"{d_k=}")
# we divided the attention_scores by sqrt(dimension) to reducing the growth of sqrt(dimension) to 1
attention_weights=torch.softmax(attention_scores/d_k**0.5, dim=-1) 
print(f"{attention_weights.shape=}\n{attention_weights}")

attention_scores.shape=torch.Size([6, 6])
tensor([[0.9231, 1.3545, 1.3241, 0.7910, 0.4032, 1.1330],
        [1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
        [1.2544, 1.8284, 1.7877, 1.0654, 0.5508, 1.5238],
        [0.6973, 1.0167, 0.9941, 0.5925, 0.3061, 0.8475],
        [0.6114, 0.8819, 0.8626, 0.5121, 0.2707, 0.7307],
        [0.8995, 1.3165, 1.2871, 0.7682, 0.3937, 1.0996]])
d_k=2
attention_weights.shape=torch.Size([6, 6])
tensor([[0.1551, 0.2104, 0.2059, 0.1413, 0.1074, 0.1799],
        [0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
        [0.1503, 0.2256, 0.2192, 0.1315, 0.0914, 0.1819],
        [0.1591, 0.1994, 0.1962, 0.1477, 0.1206, 0.1769],
        [0.1610, 0.1949, 0.1923, 0.1501, 0.1265, 0.1752],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]])


In [6]:
context_vectors=attention_weights@values
print(f"{context_vectors.shape=}\n{context_vectors}")

context_vectors.shape=torch.Size([6, 2])
tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]])


In [7]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        """
        Args:
            d_in (int): Dimension of token embeddings
            d_out (int): Dimension of output embeddings
        """
        super().__init__()
        self.W_query=nn.Parameter(torch.rand(d_in, d_out))
        self.W_key=nn.Parameter(torch.rand(d_in, d_out))
        self.W_value=nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        '''
        Args:
            x (torch.Tensor): Tensor of shape (N,d_in), where N is the number of tokens and d_in is the dimension of token embedding
        '''
        keys=x@self.W_key # (N,d_out)
        queries=x@self.W_query
        values=x@self.W_value
        attn_scores=queries @ keys.T # (N,N)
        attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5, dim=-1)
        context_vec=attn_weights@values #(N,d_out)
        print(f"{keys.max().item()=}, {queries.max().item()=}, {attn_scores.max().item()=}, {context_vec.max().item()=}")
        return context_vec

torch.manual_seed(123)
sa_v1=SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

keys.max().item()=1.1418625116348267, queries.max().item()=1.455058217048645, attn_scores.max().item()=1.8523844480514526, context_vec.max().item()=0.82103031873703
tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [8]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query=nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key=nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value=nn.Linear(d_in, d_out, bias=qkv_bias)
    def forward(self, x):
        '''
        Args:
            x (torch.Tensor): Tensor of shape (N,d_in), where N is the number of tokens and d_in is the dimension of token embedding
        '''
        keys=self.W_key(x) # (N,d_out)
        queries=self.W_query(x)
        values=self.W_value(x)
        attn_scores=queries@keys.T # (N,N)
        attn_weights=torch.softmax(attn_scores/keys.shape[1]**0.5, dim=-1)
        context_vec=attn_weights@values
        print(f"{keys.max().item()=}, {queries.max().item()=}, {attn_scores.max().item()=}, {context_vec.max().item()=}")
        return context_vec

torch.manual_seed(789)
sa_v2=SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

keys.max().item()=0.3146900534629822, queries.max().item()=0.909066915512085, attn_scores.max().item()=0.46564239263534546, context_vec.max().item()=0.07128992676734924
tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


In [9]:
state_dict=OrderedDict()
for key, value in sa_v1.state_dict().items():
    state_dict[f"{key}.weight"]=value.T

sa_v2.load_state_dict(state_dict)

print(f"{sa_v2.W_key.weight.T.shape=}, {sa_v1.W_key.shape=}")
print(f"{torch.allclose(sa_v2.W_key.weight.T, sa_v1.W_key)=}")
print(f"{torch.allclose(sa_v2.W_query.weight.T, sa_v1.W_query)=}")
print(f"{torch.allclose(sa_v2.W_value.weight.T, sa_v1.W_value)=}")

print()
print(sa_v1(inputs))
print(sa_v2(inputs))
keys1=inputs@sa_v1.W_key
keys2=inputs@sa_v2.W_key.weight.T
print(torch.allclose(sa_v1.W_key, sa_v2.W_key.weight.T), torch.allclose(keys1, keys2))

sa_v2.W_key.weight.T.shape=torch.Size([3, 2]), sa_v1.W_key.shape=torch.Size([3, 2])
torch.allclose(sa_v2.W_key.weight.T, sa_v1.W_key)=True
torch.allclose(sa_v2.W_query.weight.T, sa_v1.W_query)=True
torch.allclose(sa_v2.W_value.weight.T, sa_v1.W_value)=True

keys.max().item()=1.1418625116348267, queries.max().item()=1.455058217048645, attn_scores.max().item()=1.8523844480514526, context_vec.max().item()=0.82103031873703
tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)
keys.max().item()=1.1418625116348267, queries.max().item()=1.455058217048645, attn_scores.max().item()=1.8523844480514526, context_vec.max().item()=0.82103031873703
tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)
True True


In [10]:
with torch.no_grad():
    for p_dest, p_src in zip(sa_v2.parameters(), sa_v1.parameters()):
        p_dest.copy_(p_src.T)

print(f"{sa_v2.W_key.weight.T.shape=}, {sa_v1.W_key.shape=}")
print(f"{torch.allclose(sa_v2.W_key.weight.T, sa_v1.W_key)=}")
print(f"{torch.allclose(sa_v2.W_query.weight.T, sa_v1.W_query)=}")
print(f"{torch.allclose(sa_v2.W_value.weight.T, sa_v1.W_value)=}")

print()
print(sa_v1(inputs))
print(sa_v2(inputs))
keys1=inputs@sa_v1.W_key
keys2=inputs@sa_v2.W_key.weight.T
print(torch.allclose(sa_v1.W_key, sa_v2.W_key.weight.T), torch.allclose(keys1, keys2))

sa_v2.W_key.weight.T.shape=torch.Size([3, 2]), sa_v1.W_key.shape=torch.Size([3, 2])
torch.allclose(sa_v2.W_key.weight.T, sa_v1.W_key)=True
torch.allclose(sa_v2.W_query.weight.T, sa_v1.W_query)=True
torch.allclose(sa_v2.W_value.weight.T, sa_v1.W_value)=True

keys.max().item()=1.1418625116348267, queries.max().item()=1.455058217048645, attn_scores.max().item()=1.8523844480514526, context_vec.max().item()=0.82103031873703
tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)
keys.max().item()=1.1418625116348267, queries.max().item()=1.455058217048645, attn_scores.max().item()=1.8523844480514526, context_vec.max().item()=0.82103031873703
tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)
True True


## Causal attention

In [16]:
queries=sa_v2.W_query(inputs)
keys=sa_v2.W_key(inputs)
attn_scores=queries @ keys.T
print(f"{attn_scores.shape=}")
attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5, dim=-1)
print(f"{attn_weights=}")

# create mask
context_length=attn_scores.shape[0]
print(f"{context_length=}")
mask_simple=torch.tril(torch.ones(context_length, context_length))
print(f"{mask_simple=}")

# zero out value above diagonal
masked_simple=attn_weights*mask_simple
print(f"{masked_simple=}")

# renormalize the attention weight so each row sums to 1
row_sums=masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm=masked_simple/row_sums
print(f"{masked_simple_norm=}")
masked_simple_norm.sum(dim=-1)

attn_scores.shape=torch.Size([6, 6])
attn_weights=tensor([[0.1551, 0.2104, 0.2059, 0.1413, 0.1074, 0.1799],
        [0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
        [0.1503, 0.2256, 0.2192, 0.1315, 0.0914, 0.1819],
        [0.1591, 0.1994, 0.1962, 0.1477, 0.1206, 0.1769],
        [0.1610, 0.1949, 0.1923, 0.1501, 0.1265, 0.1752],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]],
       grad_fn=<SoftmaxBackward0>)
context_length=6
mask_simple=tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])
masked_simple=tensor([[0.1551, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1500, 0.2264, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1503, 0.2256, 0.2192, 0.0000, 0.0000, 0.0000],
        [0.1591, 0.1994, 0.1962, 0.1477, 0.0000, 0.0000],
        [0.1610, 0.1949, 0.1923, 0.1501, 0.1265, 0.0000],
        [0.1557, 0.2092, 0.2

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SumBackward1>)

In [19]:
mask=torch.triu(torch.ones(context_length, context_length), diagonal=1)
print(f"{mask=}")
masked=attn_scores.masked_fill(mask.bool(), -torch.inf)
print(f"{masked=}")
attn_weights=torch.softmax(masked/keys.shape[-1]**0.5, dim=-1)
print(f"{attn_weights=}")

mask=tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])
masked=tensor([[0.9231,   -inf,   -inf,   -inf,   -inf,   -inf],
        [1.2705, 1.8524,   -inf,   -inf,   -inf,   -inf],
        [1.2544, 1.8284, 1.7877,   -inf,   -inf,   -inf],
        [0.6973, 1.0167, 0.9941, 0.5925,   -inf,   -inf],
        [0.6114, 0.8819, 0.8626, 0.5121, 0.2707,   -inf],
        [0.8995, 1.3165, 1.2871, 0.7682, 0.3937, 1.0996]],
       grad_fn=<MaskedFillBackward0>)
attn_weights=tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3986, 0.6014, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2526, 0.3791, 0.3683, 0.0000, 0.0000, 0.0000],
        [0.2265, 0.2839, 0.2794, 0.2103, 0.0000, 0.0000],
        [0.1952, 0.2363, 0.2331, 0.1820, 0.1534, 0.0000],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]],
       grad_fn=<SoftmaxBackward0>)

In [23]:
print(f"{context_length=}")
torch.manual_seed(123)
dropout=torch.nn.Dropout(0.5) 
example=torch.ones(context_length,context_length)
print(f"{example.shape=}\n", dropout(example))

context_length=6
example.shape=torch.Size([6, 6])
 tensor([[2., 2., 2., 2., 2., 2.],
        [0., 2., 0., 0., 0., 0.],
        [0., 0., 2., 0., 2., 0.],
        [2., 2., 0., 0., 0., 2.],
        [2., 0., 0., 0., 0., 2.],
        [0., 2., 0., 0., 0., 0.]])


In [24]:
print(attn_weights, '\n')
print(dropout(attn_weights))

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3986, 0.6014, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2526, 0.3791, 0.3683, 0.0000, 0.0000, 0.0000],
        [0.2265, 0.2839, 0.2794, 0.2103, 0.0000, 0.0000],
        [0.1952, 0.2363, 0.2331, 0.1820, 0.1534, 0.0000],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]],
       grad_fn=<SoftmaxBackward0>) 

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.7366, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5677, 0.0000, 0.4206, 0.0000, 0.0000],
        [0.0000, 0.4727, 0.4662, 0.3639, 0.3068, 0.0000],
        [0.3115, 0.4183, 0.0000, 0.0000, 0.2178, 0.3588]],
       grad_fn=<MulBackward0>)


In [25]:
# simulate batch
batch=torch.stack((inputs, inputs), dim=0)
print(f"{batch.shape=}, {batch.dtype=}") # 2 input texts with 6 token each, each token is 3D embeddings


batch.shape=torch.Size([2, 6, 3]), batch.dtype=torch.float32


In [26]:
torch.triu(torch.ones(context_length, context_length), diagonal=1).shape

torch.Size([6, 6])

In [33]:
class CausalAttention(nn.Module):
    
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out=d_out
        self.W_query=nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key=nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value=nn.Linear(d_in, d_out,bias=qkv_bias)
        self.dropout=nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in=x.shape
        keys=self.W_key(x) # (B,N,d_out) where N is num_tokens, B is batch_size
        queries=self.W_query(x)
        values=self.W_value(x)

        # (B,N,N)
        attn_scores=queries@keys.transpose(1,2) # transpose dimnensions 1 and 2 while keeping the batch dimension at 0
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf) # in case, num_tokens < context_length
        attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5, dim=-1)
        attn_weights=self.dropout(attn_weights)
        context_vec=attn_weights@values
        return context_vec

torch.manual_seed(123)
context_length=batch.shape[1]
ca=CausalAttention(d_in, d_out, context_length=context_length, dropout=0.0)
context_vecs=ca(batch)
print(f'{context_vecs.shape=}')

context_vecs.shape=torch.Size([2, 6, 2])


## Multi-head

In [35]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads=nn.ModuleList([CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) for _ in range(num_heads)])

    def forward(self, x): return torch.cat([head(x) for head in self.heads], dim=-1)
        
torch.manual_seed(123)
context_length, d_in=batch.shape[1:] # the number of tokens, and embedding dimension
d_out=2
print(f"{context_length=}, {d_in=}, {d_out=}")
mha=MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs=mha(batch)
print(f"{context_vecs.shape=}\n{context_vecs}")

context_length=6, d_in=3, d_out=2
context_vecs.shape=torch.Size([2, 6, 4])
tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)


In [37]:
mha=MultiHeadAttentionWrapper(d_in, d_out=1, context_length=context_length, dropout=0.0, num_heads=2)
context_vecs=mha(batch)
print(f"{context_vecs.shape=}\n{context_vecs}")

context_vecs.shape=torch.Size([2, 6, 2])
tensor([[[0.0189, 0.2729],
         [0.2181, 0.3037],
         [0.2804, 0.3125],
         [0.2830, 0.2793],
         [0.2476, 0.2541],
         [0.2748, 0.2513]],

        [[0.0189, 0.2729],
         [0.2181, 0.3037],
         [0.2804, 0.3125],
         [0.2830, 0.2793],
         [0.2476, 0.2541],
         [0.2748, 0.2513]]], grad_fn=<CatBackward0>)


In [39]:
class MultiheadAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):

        super().__init__()
        assert (d_out%num_heads==0), "d_out must be divisible by num_heads"

        self.d_out=d_out
        self.num_heads=num_heads
        self.head_dim=d_out//num_heads # reduce the projection dim to match the desired output dim
        self.W_query=nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key=nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value=nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj=nn.Linear(d_out, d_out) # use a linear layer to combine head outputs
        self.dropout=nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self,x):
        b, num_tokens, d_in=x.shape
        keys=self.W_key(x) # (B, N, d_out) where N=num_tokens, and B is batch size
        queries=self.W_query(x)
        values=self.W_value(x)

        # We implicitly split the matrix by adding num_heads dimension
        keys=keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values=values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries=queries.view(b, num_tokens, self.num_heads, self.head_dim) # (B,N,H,Hd) where H is num_heads, and Hd is head_dim

        # swap between num_tokens and num_heads
        keys=keys.transpose(1,2)
        queries=queries.transpose(1,2) #(B,H,N,Hd) where H is num_heads, and Hd is head_dim
        values=values.transpose(1,2)

        # Compute the dot product for each head
        attn_scores=queries@keys.transpose(2,3) # (B,H,N,Hd) (B,H,Hd,N) -> (B,H,N,N)
        mask_bool=self.mask.bool()[:num_tokens, :num_tokens] # truncate mask to the number of tokens
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5, dim=-1)
        attn_weights=self.dropout(attn_weights)

        context_vec=(attn_weights@values).transpose(1,2) # (B,H,N,N)@(B,H,N,Hd)= (B,H,N,Hd) -> (B,N,H,Hd)
        context_vec=context_vec.contiguous().view(b, num_tokens, self.d_out) # combine num_heads and head_dim to d_out
        context_vec=self.out_proj(context_vec)
        return context_vec

torch.manual_seed(123)
batch_size, context_length, d_in=batch.shape
d_out=2
mha=MultiheadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs=mha(batch)
print(f"{batch.shape=}, {context_vecs.shape=}\n{context_vecs=}")

batch.shape=torch.Size([2, 6, 3]), context_vecs.shape=torch.Size([2, 6, 2])
context_vecs=tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)


In [ ]:
mha=MultiheadAttention(d_in=768, d_out=768, context_length=1024, 0.0, num_heads=12)